In [ ]:
# генерация данных для учебной e-commerce базы.
# данные не отражают реальный бизнес — цель: получить реалистичную структуру
# для практики SQL, pandas и последующей очистки данных.


from faker import Faker
from sqlalchemy import create_engine
import pandas as pd
import random
from dotenv import load_dotenv
import os

load_dotenv()  # переменные из .env файла

fake = Faker('ru_RU') # локализация для реалистичных русских имён и городов

db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

n_customers = 800
customers = []

for i in range(n_customers):
    customers.append({
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'email': fake.unique.email(), # unique гарантирует отсутствие дублей
        'phone': fake.phone_number() if random.random() > 0.2 else None,  # 20% без телефона
        'city': fake.city(),
        'signup_date': fake.date_between(start_date='-2y', end_date='today')
    })

customers_df = pd.DataFrame(customers)

# if_exists='append' — добавляем строки к уже существующей таблице,
# а не пересоздаём её (она уже создана вручную в pgAdmin)
# index=False — не записывать служебный индекс pandas как отдельную колонку
customers_df.to_sql('customers', engine, if_exists='append', index=False)

print(f"Добавлено {len(customers_df)} клиентов")

Добавлено 800 клиентов


In [ ]:
categories = ['Электроника', 'Одежда', 'Дом и сад', 'Спорт', 'Книги', 'Красота']

n_products = 200
products = []
for _ in range(n_products):
    cost = round(random.uniform(5, 300), 2)
    price = round(cost * random.uniform(1.3, 2.5), 2) # цена = себестоимость с наценкой (30-150%)
    products.append({
        'product_name': fake.catch_phrase(), # случайное название товара 
        'category': random.choice(categories),
        'price': price,
        'cost': cost,
        'is_active': random.random() > 0.1 # 10% товаров сняты с продажи
    })

products_df = pd.DataFrame(products)
products_df.to_sql('products', engine, if_exists='append', index=False)

print(f"Добавлено {len(products_df)} товаров")

Добавлено 200 товаров


In [ ]:
# customer_id и product_id сгенерированы автоматически (GENERATED ALWAYS AS IDENTITY),
# поэтому считатываем их из базы, чтобы использовать при создании orders и order_items

customer_ids = pd.read_sql("SELECT customer_id FROM customers", engine)['customer_id'].tolist()
product_ids_prices = pd.read_sql("SELECT product_id, price FROM products WHERE is_active = TRUE", engine)

In [ ]:
from datetime import datetime, timedelta

statuses = ['completed', 'shipped', 'cancelled', 'refunded']
status_weights = [0.7, 0.15, 0.1, 0.05]  # большинство заказов завершены успешно

n_orders = 4000
orders = []

for _ in range(n_orders):
    days_ago = int(random.betavariate(2, 5) * 730)  # betavariate(2, 5) даёт распределение, смещённое к недавним датам — имитация роста бизнеса со временем, а не равномерный поток заказов
    order_date = datetime.now() - timedelta(days=days_ago)
    
    orders.append({
        'customer_id': random.choice(customer_ids),
        'order_date': order_date,
        'status': random.choices(statuses, weights=status_weights)[0] # choices - выбор статуса с учётом вероятностей, определённых выше; возвращает список, поэтому [0] достаёт только первое значение
    })

orders_df = pd.DataFrame(orders)
orders_df.to_sql('orders', engine, if_exists='append', index=False)

print(f"Добавлено {len(orders_df)} заказов")

Добавлено 4000 заказов


In [ ]:
orders_full = pd.read_sql('select order_id, status from orders', engine)

order_items = []
payments = []

for _, order in orders_full.iterrows():
    n_items = random.randint(1,5) # от 1 до 5 позиций в заказе
    chosen_products = product_ids_prices.sample(n_items) # случайные товары без повторов

    order_total = 0 # сумма заказа
    for _, product in chosen_products.iterrows():
        quantity = random.randint(1,3)
        item_total = product['price'] * quantity
        order_total += item_total # накопление суммы заказа с каждым товаром

        order_items.append({
            'order_id': order['order_id'],
            'product_id': product['product_id'],
            'quantity': quantity,
            'price_at_purchase': product['price'] # цена фиксируется на момент заказа, так как стоимость может измениться, но важно указывать стоимость, по которой товар был куплен
        })

    if order['status'] != 'cancelled': # платёж создается для всех заказов, не завершённых отменой
        payments.append({
            'order_id': order['order_id'],
            'payment_date': datetime.now(), # упрощение - в реальности время оплаты было бы близко к времени заказа
            'amount': round(order_total, 2),
            'method': random.choice(['credit_card', 'paypal', 'bank_transfer', 'cash'])
        }) 

order_items_df = pd.DataFrame(order_items)
order_items_df.to_sql('order_items', engine, if_exists='append', index=False)

payments_df = pd.DataFrame(payments)
payments_df.to_sql('payments', engine, if_exists='append', index=False)

print(f"Добавлено {len(order_items_df)} позиций заказов и {len(payments_df)} платежей")   

Добавлено 12135 позиций заказов и 3637 платежей
